# V8 — Hierarchical Gated Architecture

Three-stage classifier:
1. **Hard gate**: X42 > 0.025067 OR X39 >= 169 -> predict 0 (eliminates 41% of test, zero FP)
2. **Hot-zone stacking**: LGB+XGB+CatBoost on 918 ungated rows, LR meta with Platt calibration
3. **Score-aware threshold**: maximizes (Recall+Precision)/2

Within-grade z-scores: per-X39 mean/std computed CV-safely per fold.

In [ ]:
# Stage 1: hard gate
gate = (df['X42'] > 0.025067) | (df['X39'] >= 169)
pred[gate] = 0

# Stage 2: hot-zone ensemble
hot = ~gate
p_hot = ensemble(df[hot])  # LGB + XGB + CatBoost -> LR meta + Platt

# Stage 3: threshold sweep
pred[hot] = (p_hot >= T_chosen).astype(int)
